#  Практика: Гетерогенные GNN на примере DBLP
HeteroConv + GraphSAGE и HGTConv

На этой практике мы рассмотрим, как обучать гетерогенные графовые нейронные сети на датасете DBLP с использованием PyTorch Geometric.
Мы рассмотрим две архитектуры:

- Heterogeneous GraphSAGE (через HeteroConv)

- HGT (Heterogeneous Graph Transformer)

Обе модели работают на одном и том же гетерографе и решают задачу классификации авторов (node classification).

## Гетерогенный граф: формализация

Пусть у нас есть гетерограф:

$
G = (V, E, \phi_V, \phi_E),
$

где

* $ V = \bigcup_{t \in T_V} V_t $ — множество узлов, разбитое по типам,
* $ E = \bigcup_{r \in T_E} E_r $ — множество рёбер, разбитое по типам,
* $ \phi_V: V \to T_V $ — функция отображения узлов в типы узлов,
* $ \phi_E: E \to T_E $ — функция отображения рёбер в типы рёбер.

Каждому типу узлов (t) соответствует матрица признаков:

$
X_t \in \mathbb{R}^{|V_t| \times d_t}.
$

Каждому типу рёбер $ r = (s \to t) $ соответствует матрица смежности:

$
A_r \subseteq V_s \times V_t.
$


In [ ]:
! pip install torch_geometric

In [ ]:
import torch
import torch.nn.functional as F

import torch_geometric
import torch_geometric.transforms as T
from torch_geometric.datasets import DBLP
from torch_geometric.nn import HeteroConv, Linear, SAGEConv, HGTConv


## Загрузка датасета DBLP

PyG предоставляет готовый гетерогенный датасет DBLP, содержащий узлы четырёх типов:

- author
- paper
- term
- conference

и соответствующие типы рёбер.

В примере используется трансформ:

"transform=T.Constant(node_types='conference')"


который задаёт фиксированный фичер для конференций, у которых по умолчанию нет признаков.
Это гарантирует, что все узлы имеют хотя бы один признак для обработки.

In [ ]:
# We initialize conference node features with a single one-vector as feature:
dataset = DBLP(root='data/dblp', transform=T.Constant(node_types='conference'))
data = dataset[0]

##  HeteroGNN: Гетерогенный GraphSAGE на основе HeteroConv

Модель реализована с помощью слоя `HeteroConv`, который позволяет задать **отдельный GNN-оператор для каждого типа ребра**, а затем агрегировать результаты в единое представление узлов.


### 1. GraphSAGE: напоминание для однородного графа

Классическая формула (Hamilton et al., 2017):

Узел (v) обновляет своё представление:

$
h_v^{(k)} = \sigma!\left( W^{(k)} \cdot \text{AGG}_{u \in \mathcal{N}(v)} \left( h_u^{(k-1)} \right) \right).
$

Где:

* $ \text{AGG} $ — агрегатор (mean, max, LSTM),
* $ W^{(k)} $ — обучаемая матрица слоя (k),
* $ \sigma $ — нелинейность.

---

### 2. Как обобщить GraphSAGE на гетерограф?

Проблема:
разные типы рёбер несут **разные виды отношений**, поэтому агрегировать всё одинаково неправильно.

Решение PyG:
применить **свой GraphSAGE для каждого типа рёбер**:

$
h_{v}^{(k,r)} = \text{SAGEConv}*r\left( h*{v}^{(k-1)}, {h_{u}^{(k-1)} : (u,v) \in E_r} \right).
$

А затем агрегировать вклад всех рёбер:

$
h_v^{(k)} = \sum_{r \in T_E} h_{v}^{(k,r)}
$
Именно это делает `HeteroConv` в PyG.

---

### 3. Как устроен слой HeteroConv

В модели создаётся:

```python
HeteroConv({
    edge_type: SAGEConv((-1, -1), hidden_channels)
    for edge_type in metadata[1]
})
```

Для каждого типа ребра ( r ) создаётся своя отдельная копия SAGEConv:

$
\text{SAGEConv}_r : \mathbb{R}^{d_s} \to \mathbb{R}^{d_t}.
$

Тогда полное обновление узла типа (t):

$
h_v^{(k)} = \sigma \Bigg(
\sum_{r \in \text{InEdges}(t)}
\text{SAGEConv}*r\left( h_v^{(k-1)}, {h*{u}^{(k-1)} : (u, v) \in E_r} \right)
\Bigg)
$

### Где:

* **InEdges(t)** — множество всех типов рёбер, входящих в узлы типа (t).
* Агрегатор по рёбрам — `sum` (задано в конструкторе `HeteroConv`).

---

### 4. Forward-проход модели HeteroGNN

Полный forward выглядит так:

```python
for conv in self.convs:
    x_dict = conv(x_dict, edge_index_dict)
    x_dict = {key: F.leaky_relu(x) for key, x in x_dict.items()}
```

### Формально:

Пусть $ h_t^{(k)} $ — матрица скрытых представлений узлов типа (t) на слое (k).
Тогда:

$
h_t^{(k)} = \sigma\left(
\sum_{r \in \text{InEdges}(t)}
\text{SAGEConv}_r(X_s^{(k-1)}, A_r)
\right).
$

Где:

* $ X_s^{(k-1)} $ — признаки узлов источника рёбер типа $ r=(s\to t) $,
* $ \sigma $ — LeakyReLU.

### Почему LeakyReLU?

* Устраняет проблему "мертвых" нейронов ReLU,
* Улучшает стабильность обучения на разреженных графах.

---

### 5. Финальный линейный классификатор

После $ L $ слоёв получаем финальные эмбеддинги авторов:

$
Z = h_{\text{author}}^{(L)}.
$

Затем применяется линейный слой:

$
\hat{Y} = Z W + b,
$

где:

* $ W \in \mathbb{R}^{d \times C} $ — параметры классификатора,
* $ C $ — количество классов.

В коде:

```python
return self.lin(x_dict['author'])
```



In [ ]:
class HeteroGNN(torch.nn.Module):
    def __init__(self, metadata, hidden_channels, out_channels, num_layers):
        super().__init__()

        # TODO

In [ ]:
model = HeteroGNN(data.metadata(), hidden_channels=64, out_channels=4,
                  num_layers=2)

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch_geometric.is_xpu_available():
    device = torch.device('xpu')
else:
    device = torch.device('cpu')
data, model = data.to(device), model.to(device)

In [ ]:
with torch.no_grad():  # Initialize lazy modules.
    out = model(data.x_dict, data.edge_index_dict)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=0.001)

In [ ]:
def train():
    # TODO

    return float(loss)

In [ ]:
@torch.no_grad()
def test():
    # TODO

    return accs


In [ ]:
for epoch in range(1, 21):
    loss = train()
    train_acc, val_acc, test_acc = test()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Train: {train_acc:.4f}, '
          f'Val: {val_acc:.4f}, Test: {test_acc:.4f}')

### Итоговое резюме

Модель `HeteroGNN`:

* использует стандартный **GraphSAGE** агрегатор,
* обучает **отдельные параметры для каждого типа ребра**,
* соединяет результаты через **межреляционную агрегацию (sum)**,
* применяет LeakyReLU для устойчивости,
* работает напрямую с `x_dict` и `edge_index_dict`,
* классифицирует только узлы типа **author**.

---

###  Ключевые математические идеи, которые стоит объяснить студентам

-  Гетерограф — это набор подграфов по реляционным типам

Каждый edge type определяет свою матрицу смежности (A_r).

-  Каждый тип ребра получает отдельный GraphSAGE

Это важно: **разные отношения → разные параметры**.

-  Агрегация идёт по рёбрам и по соседям

2 уровня агрегирования:

1. **межсоседское (GraphSAGE внутри конкретного edge type)**
2. **межреляционное (HeteroConv суммирует по edge types)**

-  Разные node types обучаются совместно

Гетеромодель распространяет информацию между сущностями разных типов.



# HGT
(источник - https://arxiv.org/pdf/2003.01332)

##  Архитектура HGT (Heterogeneous Graph Transformer)

HGT — это одна из самых выразительных моделей для гетерогенных графов.
Она обобщает идею self-attention (как в Transformer) на случай, когда:

* узлы имеют **разные типы**,
* рёбра имеют **разные типы**,
* каждая связь несёт **семантический смысл**, который нужно учитывать при агрегации.

---

### 1. Проблема гетерографов

В отличие от обычного графа:

```
(author) — (paper) — (term) — (conference)
```

каждый тип узла и ребра:

* имеет разную природу,
* требует своих параметров,
* несёт различный смысл в модели.

Поэтому HGT различает:

* **тип узла-источника**,
* **тип узла-получателя**,
* **тип ребра**,
* и использует разные наборы параметров для каждого из них.

---

### 2. Общая структура слоя HGTConv

Один слой HGTConv состоит из:

1. **Type-specific Q/K/V проекторов**
2. **Relation-specific трансформаций для ребра**
3. **Multi-head attention**
4. **Агрегации сообщений**
5. **Residual + LayerNorm**
6. **Feed-forward блока**
7. **Ещё одного Residual + LayerNorm**

Это полностью аналогично Transformer, но адаптировано к гетерографам.

---

### 3. Type-specific Q/K/V (разные для каждого типа узла)

Каждый тип узла имеет свои матрицы для Query, Key и Value.

Например:

```
W_Q["author"],  W_K["author"],  W_V["author"]
W_Q["paper"],   W_K["paper"],   W_V["paper"]
W_Q["term"],    W_K["term"],    W_V["term"]
...
```

Это означает:

* авторы формируют свои собственные Q/K/V,
* статьи — свои,
* термы — свои.

Таким образом, модель понимает, что разные типы узлов — это разные сущности.

---

### 4. Relation-specific параметры для каждого типа ребра

Каждый тип ребра (например, `paper → author`) имеет **свой набор параметров**:

```
W_rel["paper->author"]
W_rel["author->paper"]
W_rel["paper->term"]
...
```

Эти параметры изменяют Key и Value при передаче сообщений:

```
K = W_K[source_node_type] * h_source
K_rel = W_rel[edge_type] * K

V = W_V[source_node_type] * h_source
V_rel = W_rel[edge_type] * V
```

Связь `author → paper` интерпретируется иначе, чем `paper → term`,
поскольку их семантика полностью различается.

---

### 5. Attention между узлами

Для каждого ребра `u → v` attention вычисляется как:

```
score(u → v) = dot( Q_v , K_rel_u ) / sqrt(d)
```

Затем по всем соседям применяется softmax:

```
α(u → v) = softmax(score(u → v) across all neighbors u)
```

Где:

* `Q_v` — Query целевого узла,
* `K_rel_u` — Key источника, преобразованный с учётом типа ребра,
* `α(u → v)` — важность соседа `u` при обновлении узла `v`.

---

### 6. Передача сообщений (Message Passing)

Каждое сообщение от соседа `u` к узлу `v` вычисляется как:

```
message(u → v) = α(u → v) * V_rel_u
```

То есть attention определяет, какие узлы важнее.

---

### 7. Агрегация по типам рёбер

Узел может получать сообщения от разных типов рёбер.

HGT суммирует их:

```
messages_v = sum_over_edge_types ( sum_over_neighbors_in_this_edge_type ( message ) )
```

Так учитывается структура всего гетерографа, а не только отдельных реляций.

---

### 8. Residual + Normalization

После агрегации:

```
h_v_new = LayerNorm( h_v_old + messages_v )
```

Это важно для стабильного обучения и позволяет строить глубокие модели.

---

### 9. Feed-Forward блок (как в Transformer)

Каждый узел проходит через небольшой MLP:

```
FFN(h) = Linear2( ReLU( Linear1(h) ) )
```

После FFN снова применяется residual и LayerNorm:

```
h_v_out = LayerNorm( h_v_new + FFN(h_v_new) )
```

---

### 10. Multi-Head Attention

HGT использует **несколько attention голов**, как и стандартный Transformer.

Каждая голова имеет свои наборы параметров:

```
W_Q_head_i[type]
W_K_head_i[type]
W_V_head_i[type]
W_rel_head_i[edge_type]
```

Разные головы выделяют разные аспекты графа:

* одна может фокусироваться на авторских связях,
* другая — на связях между статьями и термами,
* третья — на конференциях.

Итоговая репрезентация — конкатенация или сумма по головам.

---

### 11. Итоговая схема работы одного слоя HGTConv

Каждый слой выполняет следующие шаги:

1. Построение Q/K/V для всех узлов разных типов
2. Реляционное преобразование Key/Value в зависимости от типа ребра
3. Attention между узлами
4. Агрегация сообщений
5. Residual + LayerNorm
6. Feed-forward блок
7. Residual + LayerNorm
8. Выходные векторы для каждого типа узлов

Это делает HGT **универсальным Transformer-блоком для гетерографов**.



In [ ]:
class HGT(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels, num_heads, num_layers):
        super().__init__()

        # TODO

In [ ]:
model = HGT(hidden_channels=64, out_channels=4, num_heads=2, num_layers=1)

In [ ]:
data, model = data.to(device), model.to(device)

In [ ]:
with torch.no_grad():  # Initialize lazy modules.
    out = model(data.x_dict, data.edge_index_dict)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=0.001)



In [ ]:
def train():
    # TODO


@torch.no_grad()
def test():
    # TODO


In [ ]:
for epoch in range(1, 21):
    loss = train()
    train_acc, val_acc, test_acc = test()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Train: {train_acc:.4f}, '
          f'Val: {val_acc:.4f}, Test: {test_acc:.4f}')

### Почему HGT работает лучше обычных GNN?

- Разные параметры для разных типов узлов

→ модель различает смысл сущностей.

- Разные параметры для разных типов рёбер

→ модель различает смысл отношений.

-  Attention как механизм важности

→ модель выбирает наиболее значимые соседние узлы.

-  Multi-head архитектура

→ несколько способов анализа структуры графа одновременно.

-  Transformer-блок

→ позволяет строить глубокие, стабильные модели.



### Где HGT особенно эффективен?

* научные графы (DBLP, ACM, MAG)
* рекомендательные системы (user/item)
* биоинформатика (gene/protein interactions)
* графы знаний (knowledge graphs)
* многорелированные социальные сети



еще больше примеров - https://github.com/pyg-team/pytorch_geometric/tree/master/examples/hetero